# Day 06 — OOP Part I: Classes, @property, __repr__, isinstance

==============================================================
Module 01: Python + Async + FastAPI for LLM Engineering
Vidya Sankalp | Applied GenAI Engineering

WHAT THIS FILE COVERS:
  - class, __init__, self, instance methods
  - @property — computed attributes (the Pythonic way)
  - __repr__ — what prints when you inspect an object
  - isinstance() — type checking at runtime

WHY THIS MATTERS FOR LLM ENGINEERING:
  - PromptBuilder encapsulates prompt template logic
  - LLMClient encapsulates API call + config
  - @property exposes token count, validation status without calling methods
  - isinstance() is used throughout LangChain for type dispatch


In [ ]:

import json
import logging
from typing import Any

log = logging.getLogger(__name__)




## CLASS 1: PromptBuilder

Encapsulates the 5-section prompt template from Module 03
═════════════════════════════════════════════════════════════════════════════


In [ ]:

class PromptBuilder:
    """
    Builds structured LLM system prompts from component parts.

    Instead of assembling prompts as ad-hoc f-strings scattered
    throughout the codebase, PromptBuilder centralises all prompt
    logic in one class. This makes prompts:
    - Testable (unit test the render() output)
    - Versioned (track changes to the class in git)
    - Composable (subclasses can specialise the prompt)

    Attributes:
        role          : Who the LLM is playing (e.g. "customer support agent")
        domain        : Business context (e.g. "ShopSmart e-commerce platform")
        _rules        : List of MUST/MUST NOT rules (private — use add_rule())
        output_format : Expected response structure
    """

    def __init__(
        self,
        role: str,
        domain: str,
        output_format: str = "plain text",
    ) -> None:
        """
        Initialise a PromptBuilder.

        Args:
            role         : Role description for the LLM.
            domain       : Business domain context.
            output_format: How the LLM should format its response.
        """

        # Instance variables — each instance gets its own copy of these
        # The naming convention self._rules (single underscore) means
        # "treat as private" — not enforced by Python, but a convention
        self.role          = role
        self.domain        = domain
        self.output_format = output_format
        self._rules: list[str] = []         # start with empty rules list
        self._examples: list[dict] = []     # start with empty examples list

    def add_rule(self, rule: str) -> "PromptBuilder":
        """
        Add a MUST/MUST NOT rule to the prompt.

        Returns self so calls can be chained:
            builder.add_rule("rule 1").add_rule("rule 2")

        Args:
            rule: The rule text (e.g. "You MUST NOT make up prices")

        Returns:
            self — for method chaining
        """

        self._rules.append(rule)
        return self   # returning self enables method chaining

    def add_example(self, input_text: str, output_text: str) -> "PromptBuilder":
        """
        Add a few-shot example to the prompt.

        Args:
            input_text : Example user input.
            output_text: Expected model output for that input.

        Returns:
            self — for method chaining
        """

        self._examples.append({
            "input" : input_text,
            "output": output_text,
        })
        return self

    def render(self) -> str:
        """
        Render the complete system prompt string.

        This is the main output method — call it when you are ready
        to send the prompt to the LLM API.

        Returns:
            A fully formatted system prompt string.
        """

        # Build the rules block
        if self._rules:
            rules_block = "\n".join(f"You MUST {r}" if not r.startswith("You") else r
                                    for r in self._rules)
        else:
            rules_block = "(no specific rules defined)"

        # Build the examples block
        if self._examples:
            ex_lines = []
            for ex in self._examples:
                ex_lines.append(f"Input : {ex['input']}")
                ex_lines.append(f"Output: {ex['output']}")
                ex_lines.append("")
            examples_block = "\n".join(ex_lines)
        else:
            examples_block = "(no examples defined)"

        return (
            f"## Role\n"
            f"You are a {self.role} for {self.domain}.\n\n"
            f"## Task\n"
            f"{rules_block}\n\n"
            f"## Output Format\n"
            f"{self.output_format}\n\n"
            f"## Examples\n"
            f"{examples_block}"
        )

    # ── @property — computed attributes ───────────────────────────────────────
    #
    # @property turns a method into an attribute.
    # Caller writes:  builder.char_count       (not builder.char_count())
    # Python runs :   the get_char_count logic internally
    #
    # Use @property when:
    #   - The value is derived from other attributes (computed)
    #   - You want read-only access (no setter defined)
    #   - The name reads better as a noun than as a method call
    #
    # Do NOT use @property for expensive operations (use a method instead)

    @property
    def char_count(self) -> int:
        """
        Return the character count of the rendered prompt.

        This is a @property because:
        - It's derived from calling render()
        - It reads naturally as an attribute: builder.char_count
        - It's cheap to compute
        """
        return len(self.render())

    @property
    def rule_count(self) -> int:
        """Return how many rules are currently in the prompt."""
        return len(self._rules)

    @property
    def is_valid(self) -> bool:
        """
        Return True if the prompt has the minimum required content.

        A valid prompt must have:
        - A non-empty role
        - At least one rule
        - An output format defined
        """
        return (
            bool(self.role)             # non-empty string is truthy
            and len(self._rules) >= 1
            and bool(self.output_format)
        )

    # ── __repr__ — object representation ──────────────────────────────────────
    #
    # __repr__ defines what you see when you:
    #   - print(builder)
    #   - inspect builder in a debugger
    #   - log the object: log.info(f"Builder: {builder}")
    #
    # Without __repr__: <__main__.PromptBuilder object at 0x7f3a...> (useless)
    # With __repr__   : PromptBuilder(role='support agent', rules=3, chars=512)
    #
    # Convention: __repr__ should look like the constructor call if possible,
    # or at minimum show the key identifying fields.

    def __repr__(self) -> str:
        return (
            f"PromptBuilder("
            f"role={self.role!r}, "          # !r adds quotes around strings
            f"rules={self.rule_count}, "
            f"examples={len(self._examples)}, "
            f"chars={self.char_count}"
            f")"
        )




## CLASS 2: ProductReview — working with real data from reviews.csv

═════════════════════════════════════════════════════════════════════════════


In [ ]:

class ProductReview:
    """
    Represents a single product review from the e-commerce dataset.

    This class demonstrates:
    - __init__ with type conversion (CSV values come in as strings)
    - @property for derived attributes (star rating display)
    - __repr__ for easy debugging
    - Class methods for creating instances from different data sources

    Real data from reviews.csv:
        review_id, product_id, customer_id, rating, review_title,
        review_text, review_date, verified_purchase, helpful_votes
    """

    # Class variable — shared across ALL instances of ProductReview
    # (Instance variables belong to one object; class variables belong to the class)
    VALID_RATINGS: tuple = (1, 2, 3, 4, 5)

    def __init__(
        self,
        review_id  : int,
        product_id : int,
        customer_id: int,
        rating     : int,
        title      : str,
        text       : str,
        verified   : bool = False,
    ) -> None:
        # Validate rating at construction time — fail fast
        if rating not in self.VALID_RATINGS:
            raise ValueError(
                f"Invalid rating: {rating}. Must be one of {self.VALID_RATINGS}"
            )

        self.review_id   = review_id
        self.product_id  = product_id
        self.customer_id = customer_id
        self.rating      = rating
        self.title       = title.strip()
        self.text        = text.strip()
        self.verified    = verified

    @property
    def stars(self) -> str:
        """Display rating as filled/empty stars. e.g. ★★★☆☆"""
        return "★" * self.rating + "☆" * (5 - self.rating)

    @property
    def is_positive(self) -> bool:
        """Return True if rating is 4 or 5."""
        return self.rating >= 4

    @property
    def summary(self) -> str:
        """Return a one-line summary suitable for prompt injection."""
        verified_badge = "[Verified] " if self.verified else ""
        return f"{self.stars} {verified_badge}{self.title} — {self.text[:80]}..."

    def to_prompt_context(self) -> str:
        """
        Format this review as an XML context block for an LLM prompt.

        Returns:
            A formatted string ready to inject into the <context> section.
        """
        return (
            f"<review id='{self.review_id}'>\n"
            f"  Rating   : {self.stars} ({self.rating}/5)\n"
            f"  Verified : {'Yes' if self.verified else 'No'}\n"
            f"  Title    : {self.title}\n"
            f"  Text     : {self.text[:200]}\n"
            f"</review>"
        )

    @classmethod
    def from_csv_row(cls, row: dict) -> "ProductReview":
        """
        Create a ProductReview from a raw CSV row dict.

        @classmethod uses `cls` instead of `self`.
        `cls` refers to the class itself (ProductReview).
        This lets subclasses override from_csv_row and still work correctly.

        Args:
            row: A dict from csv.DictReader (all values are strings).

        Returns:
            A ProductReview instance.
        """
        return cls(
            review_id   = int(row["review_id"]),
            product_id  = int(row["product_id"]),
            customer_id = int(row["customer_id"]),
            rating      = int(row["rating"]),
            title       = row["review_title"],
            text        = row["review_text"],
            verified    = row.get("verified_purchase", "No").strip() == "Yes",
        )

    def __repr__(self) -> str:
        return (
            f"ProductReview("
            f"id={self.review_id}, "
            f"product={self.product_id}, "
            f"rating={self.rating}, "
            f"verified={self.verified}"
            f")"
        )




## SECTION 3: isinstance() — type checking

═════════════════════════════════════════════════════════════════════════════


In [ ]:

def process_review_or_raw(item: Any) -> str:
    """
    Handle input that might be a ProductReview or a raw dict.

    isinstance() checks if an object is an instance of a class (or subclass).
    It is safer than type(obj) == SomeClass because isinstance() also returns
    True for subclasses (type() does not).

    This pattern appears throughout LangChain when functions need to handle
    multiple input types (e.g. a string or a Document object).

    Args:
        item: Either a ProductReview instance or a raw dict from the database.

    Returns:
        A formatted context string for the LLM prompt.
    """

    if isinstance(item, ProductReview):
        # We have a fully-typed object — use its methods directly
        log.debug(f"Processing ProductReview: {item}")
        return item.to_prompt_context()

    elif isinstance(item, dict):
        # We have a raw dict — need to extract fields manually
        log.debug("Processing raw dict — consider using ProductReview.from_csv_row()")
        rating = item.get("rating", 0)
        title  = item.get("review_title", "Unknown")
        text   = item.get("review_text", "")
        stars  = "★" * int(rating) + "☆" * (5 - int(rating))
        return f"<review>\n  Rating: {stars}\n  Title: {title}\n  Text: {text[:80]}\n</review>"

    elif isinstance(item, str):
        # We have a plain string — wrap it as-is
        return f"<review>{item}</review>"

    else:
        raise TypeError(
            f"Expected ProductReview, dict, or str. Got: {type(item).__name__}"
        )




In [ ]:
if __name__ == "__main__":
    logging.basicConfig(level=logging.DEBUG, format="%(levelname)s %(message)s")

    print("=" * 60)
    print("DAY 06 — OOP I: Classes, @property, __repr__, isinstance")
    print("=" * 60)

    # PromptBuilder
    print("\n[1] PromptBuilder — method chaining")
    builder = (
        PromptBuilder(
            role="customer support specialist",
            domain="ShopSmart e-commerce platform",
            output_format="JSON: {category, confidence, response}",
        )
        .add_rule("You MUST NOT make up order details")
        .add_rule("You MUST always include the order ID in responses")
        .add_example("Where is order 3042?", '{"category": "TRACK_ORDER", "response": "Order 3042 is In Transit"}')
    )

    print(f"  repr: {builder}")
    print(f"  char_count : {builder.char_count}")
    print(f"  rule_count : {builder.rule_count}")
    print(f"  is_valid   : {builder.is_valid}")
    print(f"\n  Rendered prompt (first 200 chars):")
    print(f"  {builder.render()[:200]}...")

    # ProductReview
    print("\n[2] ProductReview — from CSV row")
    row = {
        "review_id": "5001", "product_id": "2033", "customer_id": "1210",
        "rating": "4", "review_title": "Really good value",
        "review_text": "Excellent product for the price. Works perfectly out of the box.",
        "verified_purchase": "Yes",
    }
    review = ProductReview.from_csv_row(row)
    print(f"  repr    : {review}")
    print(f"  stars   : {review.stars}")
    print(f"  positive: {review.is_positive}")
    print(f"  summary : {review.summary}")

    # isinstance
    print("\n[3] isinstance() dispatch")
    for item in [review, {"rating": 3, "review_title": "OK", "review_text": "Fine."}, "Raw string review"]:
        result = process_review_or_raw(item)
        print(f"  type({type(item).__name__:15s}) → {result[:60]}...")


## Run the demo

Run the cell below to execute the `if __name__ == '__main__':` block.


In [ ]:
# Execute the demo for this day
import runpy
runpy.run_path('modules/day06_oop_classes.py', run_name='__main__')
